# Sophia AGI — LoRA training (Google Colab)

## ⚠️ Enable GPU **before** running code

1. Menu **Runtime** → **Change runtime type**
2. **Hardware accelerator:** `T4 GPU` (free tier) or `L4 GPU`
3. Click **Save**
4. **Runtime** → **Restart session**
5. Run cells from the top

Train **Sophia** adapter on the provenance corpus with QLoRA (4-bit).

**Repo:** [github.com/tomyimkc/sophia-agi](https://github.com/tomyimkc/sophia-agi)

In [ ]:
import torch

if not torch.cuda.is_available():
    from IPython.display import display, Markdown
    display(Markdown("""
### ❌ No GPU detected

You are on **CPU runtime**. Training will not work.

1. **Runtime** → **Change runtime type**
2. Hardware accelerator: **T4 GPU**
3. **Save** → **Runtime** → **Restart session**
4. Re-run this cell — you should see a GPU name (e.g. Tesla T4)
    """))
    raise SystemExit("Stop: enable GPU runtime, restart, re-run")

print("✅ GPU:", torch.cuda.get_device_name(0))
print("   VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import os
if not os.path.isdir("sophia-agi"):
    !git clone https://github.com/tomyimkc/sophia-agi.git
else:
    print("sophia-agi already cloned")
%cd sophia-agi

In [ ]:
!pip install -q -r requirements-lora.txt bitsandbytes accelerate

In [ ]:
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL = "Qwen/Qwen2.5-7B-Instruct" if vram_gb > 20 else "Qwen/Qwen2.5-3B-Instruct"
print(f"Using {MODEL} ({vram_gb:.1f} GB VRAM)")

In [ ]:
!python tools/prepare_lora_dataset.py
!python tools/train_lora.py --4bit --epochs 3 --model {MODEL} --batch-size 2

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

ckpt = Path("training/lora/checkpoints/sophia-v1")
required = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in required if not (ckpt / f).exists()]
if missing:
    raise SystemExit(f"Training incomplete — missing {missing}. Re-run train cell.")
print("Checkpoint OK:", list(ckpt.glob("*")))

!python tools/claude_model_lab.py write-modelfile --adapter training/lora/checkpoints/sophia-v1
shutil.make_archive("sophia-lora-v1", "zip", str(ckpt))
print("Zip size MB:", round(Path("sophia-lora-v1.zip").stat().st_size / 1e6, 2))
files.download("sophia-lora-v1.zip")

## After download (local)

```bash
unzip sophia-lora-v1.zip -d training/lora/checkpoints/sophia-v1
python tools/eval_local_model.py --adapter training/lora/checkpoints/sophia-v1 --with-gate
ollama create sophia-7b -f models/ollama/Modelfile
```

Always run **sophia_gate_check** on production answers.